# Daily norms methods

### Imports

In [17]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
import polars as pl
from datetime import date
from statsmodels.nonparametric.smoothers_lowess import lowess
import numpy as np

import climato.config as config
from climato.sources.meteo_france import daily
from climato.utils.datetime import utc_now_iso

In [19]:
DATA_DIR = config.get_data_dir()
RAW_DATA_DIR = config.get_raw_data_dir()
MANIFEST_DIR = DATA_DIR / "metadata" / "manifests"
MANIFEST_PATH = MANIFEST_DIR / "daily_files.parquet"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DAILY_DIR = PROCESSED_DIR / daily.MF_DAILY_RELATIVE_PATH

### Parameter norms

In [5]:
REF_START = date(1991, 1, 1)
REF_END = date(2020, 12, 31)

In [33]:
from pathlib import Path


station_ids = list(
    pl.read_csv(
        Path(config.PROJECT_ROOT / "dump" / "stations_candidates_20261018T183248Z.csv"),
        separator=";",
        schema_overrides={
            "station_id": pl.String,
        },
    )
    .select("station_id")
    .sort("station_id")
    .to_series()
)

print(station_ids)

departments = sorted({station_id[:2] for station_id in station_ids})

['01089001', '01235001', '02320001', '03059001', '03060001', '03100001', '03108001', '03130002', '03204001', '03240001', '03310001', '03321001', '04039001', '04049001', '05046001', '05142001', '06029001', '06088001', '07010001', '07019005', '07064001', '07066001', '07068001', '07075001', '07105001', '07117002', '07128001', '07159001', '07168001', '07261001', '07265001', '07331001', '07338001', '08419001', '09181003', '09289001', '10030001', '10057001', '10130001', '10323001', '11004001', '11012001', '11069001', '11076001', '11206003', '12145001', '12253001', '12254001', '13001009', '13004003', '13047001', '13054001', '13103001', '14137001', '14239001', '14578001', '14624001', '15014004', '15050001', '15118002', '15119001', '16089001', '17218001', '17224001', '17243002', '17268002', '17323001', '17353001', '17415003', '18033001', '18057003', '18092001', '18242001', '18279001', '19125001', '19136002', '19272001', '19275001', '20004002', '20041001', '20050001', '20148001', '20247001', '20

In [6]:
station_ids = (
    '06088001', '13054001', '14137001', '16089001', '20148001', '21473001', '25056001', '26198001',
    '29075001', '30189001', '31069001', '33281001', '35281001', '36063001', '44020001', '45055001',
    '47091001', '51183001', '51449002', '54526001',
    '58160001', '59343001', '63113001', '64549001', '66136001', '67124001', '69029001', '72181001',
    '73054001', '75114001', '86027001'
)

departments = sorted({station_id[:2] for station_id in station_ids})

In [7]:
param = "tx"

is_ref = pl.col("obs_date").is_between(REF_START, REF_END)

df_ref = (
    pl.scan_parquet(
        PROCESSED_DAILY_DIR / "parameter_family=rr_t_vent",
        hive_partitioning=True,
        hive_schema={
            "dataset_year": pl.UInt16,
            "department": pl.String,
            "collection": pl.String,
            "period": pl.String,
        },
    )
    .filter(
        pl.col("collection") == "quot",
        pl.col("department").is_in(departments),
        pl.col("station_id").is_in(station_ids),
        # pl.col("obs_date").is_between(REF_START, REF_END),
    )
    .select(
        "station_id",
        "station_name",
        "obs_date",
        pl.col(param),
    )
    .sort(
        "station_id",
        "obs_date"
    )
    # .with_columns(
    #     pl.col("obs_date").dt.year().alias("year"),
    #     pl.col("obs_date").dt.month().alias("month"),
    #     pl.col("obs_date").dt.day().alias("day"),
    #     pl.col("obs_date").dt.strftime(format="%m-%d").alias("clim_day"),
    # )
    # .with_columns(
    #     pl.date(year=pl.lit(2000), month=pl.col("month"), day=pl.col("day")).alias("plot_date"),
    # )
    # .explain(optimized=True)
    .collect(engine="streaming")
)

# lf_norm_test = (
#     lf_source_dataset
#     # .filter(
#     #     pl.col("department") == "04",
#     #     pl.col("station_id") == "04039001",
#     #     # pl.col("obs_date").is_between(REF_START, REF_END),
#     # )
#     .with_columns(
#         pl.col("obs_date").dt.month().alias("month"),
#         pl.col("obs_date").dt.day().alias("day"),
#         pl.col("obs_date").dt.strftime(format="%m-%d").alias("clim_day"),
#     )
#     .group_by("month", "day", "clim_day", "plot_date")
#     .agg(
#         n_rows=pl.col(param).filter(is_ref).is_not_null().sum() + pl.col(param).filter(is_ref).is_null().sum(),
#         n_values=pl.col(param).filter(is_ref).count(),
#         norm=pl.col(param).filter(is_ref).mean(),
#         std_norm=pl.col(param).filter(is_ref).std(),
#         min_value=pl.col(param).min(),
#         max_value=pl.col(param).max(),
#     )
#     .sort(["month", "day"])
# )

# print(lf_norm_test.explain(optimized=True))

# df_norm_test = lf_norm_test.collect()

with pl.Config(tbl_cols=-1, tbl_rows=100, tbl_width_chars=200):
    print(df_ref)

shape: (1_056_338, 4)
┌────────────┬────────────────┬────────────┬──────┐
│ station_id ┆ station_name   ┆ obs_date   ┆ tx   │
│ ---        ┆ ---            ┆ ---        ┆ ---  │
│ str        ┆ str            ┆ date       ┆ f64  │
╞════════════╪════════════════╪════════════╪══════╡
│ 06088001   ┆ NICE           ┆ 1942-07-01 ┆ 27.8 │
│ 06088001   ┆ NICE           ┆ 1942-07-02 ┆ 24.6 │
│ 06088001   ┆ NICE           ┆ 1942-07-03 ┆ 25.4 │
│ 06088001   ┆ NICE           ┆ 1942-07-04 ┆ 25.0 │
│ 06088001   ┆ NICE           ┆ 1942-07-05 ┆ 28.1 │
│ 06088001   ┆ NICE           ┆ 1942-07-06 ┆ 28.7 │
│ 06088001   ┆ NICE           ┆ 1942-07-07 ┆ 27.0 │
│ 06088001   ┆ NICE           ┆ 1942-07-08 ┆ 26.7 │
│ 06088001   ┆ NICE           ┆ 1942-07-09 ┆ 26.1 │
│ 06088001   ┆ NICE           ┆ 1942-07-10 ┆ 26.7 │
│ 06088001   ┆ NICE           ┆ 1942-07-11 ┆ 33.1 │
│ 06088001   ┆ NICE           ┆ 1942-07-12 ┆ 24.8 │
│ 06088001   ┆ NICE           ┆ 1942-07-13 ┆ 24.1 │
│ 06088001   ┆ NICE           ┆ 1942-07-14

In [8]:
stations_dfs = df_ref.partition_by(
    "station_id",
    as_dict=True,
)

In [9]:
clim_days = (
    pl.DataFrame(
        pl.date_range(date(2000, 1, 1), date(2000, 12, 31), eager=True).alias("date")
    )
    .with_row_index("clim_day_idx")
    .with_columns(
        clim_day=pl.col("date").dt.strftime(format="%m-%d"),
        month=pl.col("date").dt.month(),
        day=pl.col("date").dt.day(),
    )
)

print(clim_days)

shape: (366, 5)
┌──────────────┬────────────┬──────────┬───────┬─────┐
│ clim_day_idx ┆ date       ┆ clim_day ┆ month ┆ day │
│ ---          ┆ ---        ┆ ---      ┆ ---   ┆ --- │
│ u32          ┆ date       ┆ str      ┆ i8    ┆ i8  │
╞══════════════╪════════════╪══════════╪═══════╪═════╡
│ 0            ┆ 2000-01-01 ┆ 01-01    ┆ 1     ┆ 1   │
│ 1            ┆ 2000-01-02 ┆ 01-02    ┆ 1     ┆ 2   │
│ 2            ┆ 2000-01-03 ┆ 01-03    ┆ 1     ┆ 3   │
│ 3            ┆ 2000-01-04 ┆ 01-04    ┆ 1     ┆ 4   │
│ 4            ┆ 2000-01-05 ┆ 01-05    ┆ 1     ┆ 5   │
│ …            ┆ …          ┆ …        ┆ …     ┆ …   │
│ 361          ┆ 2000-12-27 ┆ 12-27    ┆ 12    ┆ 27  │
│ 362          ┆ 2000-12-28 ┆ 12-28    ┆ 12    ┆ 28  │
│ 363          ┆ 2000-12-29 ┆ 12-29    ┆ 12    ┆ 29  │
│ 364          ┆ 2000-12-30 ┆ 12-30    ┆ 12    ┆ 30  │
│ 365          ┆ 2000-12-31 ┆ 12-31    ┆ 12    ┆ 31  │
└──────────────┴────────────┴──────────┴───────┴─────┘


In [10]:
import sys

from climato.utils.colors import Color


def circular_lowess_span(values, span_days=31, it=0):
    y = np.asarray(values, dtype=float)
    n = len(y)
    x = np.arange(n, dtype=float)

    valid_original = np.isfinite(y)
    n_valid = valid_original.sum()

    if n_valid < span_days:
        raise ValueError(
            f"Not enough valid climatological days: "
            f"n_valid={n_valid} < span_days={span_days}"
        )

    x_ext = np.concatenate([x - n, x, x + n])
    y_ext = np.concatenate([y, y, y])

    valid_ext = np.isfinite(y_ext)

    x_fit = x_ext[valid_ext]
    y_fit = y_ext[valid_ext]

    if not np.isfinite(x_fit).all():
        raise ValueError("Non-finite x_fit")
    if not np.isfinite(y_fit).all():
        raise ValueError("Non-finite y_fit")
    if not np.isfinite(x_ext).all():
        raise ValueError("Non-finite xvals/x_ext")
    
    frac = min(span_days / len(y_fit), 1.0)

    smooth_ext = lowess(
        endog=y_fit,
        exog=x_fit,
        xvals=x_ext,
        frac=frac,
        it=it,
        is_sorted=True,
        # return_sorted=False, # ignored if xvals provided
        missing="none",
    )

    return smooth_ext[n:2*n]


def log_var_lowess_span(y, span_days=31, it=0):
    var = y ** 2
    epsilon = 1e-6
    log_var = np.log(np.maximum(var, epsilon))
    log_var_smooth = circular_lowess_span(log_var, span_days=span_days, it=it)
    std_smooth = np.sqrt(np.exp(log_var_smooth))

    return std_smooth


def compute_circular_residual_std(values, window_size=7):
    """
    Compute standard deviation based on residuals.
    """
    if window_size % 2 == 0:
        raise ValueError("window_size doit être impair pour une fenêtre centrée.")
    
    half = window_size // 2

    # On colle la fin au début et le début à la fin pour gérer la circularité.
    padded = np.concatenate([
        values[-half:],
        values,
        values[:half],
    ])

    residuals_std = []
    for i in range(len(padded)):
        if i >= len(values):
            break

        window_residuals = []
        for j in range(window_size):
            window_residuals.extend(padded[i + j])

        residuals_std.append(float(np.std(np.array(window_residuals), ddof=1)))
    
    return residuals_std    


def compute_df_norm_train(df_train: pl.DataFrame):
    df_norm_train = (
        clim_days
        .join(
            df_train
            .group_by(["month", "day", "clim_day"])
            .agg(
                n_values=pl.col(param).count(),
                norm_train=pl.col(param).mean(),
                std_norm_train=pl.col(param).std(),
            ),
            how="left",
            on=["month", "day", "clim_day"],
        )
        .sort(["month", "day"])
    )


    nulls_or_nans = df_norm_train.filter(
        pl.col("std_norm_train").is_null() |
        pl.col("std_norm_train").is_nan(),
    )

    if nulls_or_nans.height:
        print(Color.RED)
        print(nulls_or_nans)
        print(Color.RESET)

    # # TODO Debug
    # sys.exit(0)

    # Méthode A
    df_norm_train = df_norm_train.with_columns(
        pl.Series(
            "mean_lowess_45d", 
            circular_lowess_span(df_norm_train["norm_train"].to_numpy(), span_days=45).tolist(),
        ),
    )

    df_norm_train = df_norm_train.with_columns(
        pl.Series(
            "std_lowess_45d",
            log_var_lowess_span(df_norm_train["std_norm_train"].to_numpy(), span_days=45).tolist(),
        ),
    )

    # Méthode B
    df_residuals = (
        df_train
        .join(
            (
                df_norm_train
                .select("clim_day", "mean_lowess_45d")
            ),
            on="clim_day",
        )
        .select(
            "clim_day",
            (pl.col(param) - pl.col("mean_lowess_45d")).alias("residual_lowess_45d"),
        )
        .group_by("clim_day")
        .agg(
            residuals_lowess_45d=pl.col("residual_lowess_45d"),
        )
        .with_columns(
            len_residuals=pl.col("residuals_lowess_45d").list.len(),
        )
        .sort("clim_day")
    )

    df_residuals = (
        clim_days
        .join(
            df_residuals,
            how="left",
            on="clim_day",
        )
    )


    df_norm_train = df_norm_train.with_columns(
        pl.Series(
            "std_residuals_7d_train", 
            compute_circular_residual_std(
                df_residuals["residuals_lowess_45d"], 
                window_size=7,
            ),
        )
    )

    df_norm_train = df_norm_train.with_columns(
        pl.Series(
            "std_lowess_45d_residuals_7d",
            log_var_lowess_span(df_norm_train["std_residuals_7d_train"].to_numpy(), span_days=45).tolist(),
        ),
    )

    # with pl.Config(tbl_cols=20, tbl_rows=100, tbl_width_chars=400):
    #     print(df_norm_train)

    # with pl.Config(tbl_cols=20, tbl_rows=100, tbl_width_chars=400):
    #     print(df_residuals)

    # sys.exit(0)

    return df_norm_train

#### Cross-validation

In [13]:
ts = utc_now_iso()
years = list(range(REF_START.year, REF_END.year + 1))
cv_summaries = []

# for station_id in station_ids[:2]:
for station_id in station_ids[:]:
    # if station_id[:2] != "51":
    #     continue
    
    base_ref = (
        # df_ref
        # .filter(pl.col("station_id") == station_id)
        stations_dfs[(station_id,)]
        .select(
            "station_id", 
            "station_name", 
            "obs_date",
            pl.col("obs_date").dt.year().alias("year"),
            pl.col("obs_date").dt.month().alias("month"),
            pl.col("obs_date").dt.day().alias("day"),
            pl.col("obs_date").dt.strftime(format="%m-%d").alias("clim_day"),
            pl.col(param),
        )
        .filter(
            pl.col("obs_date").is_between(REF_START, REF_END),
            pl.col(param).is_not_null(),
        )
    )

    # print(base_ref)

    rows = []
    to_concat_evaluated = []

    for test_year in years:
        train_df = base_ref.filter(pl.col("year") != test_year)
        test_df = base_ref.filter(pl.col("year") == test_year)

        if test_df.height == 0:
            continue

        df_norm_train = compute_df_norm_train(train_df)

        # print(df_norm_train)

        for method in ("lowess_45d", "lowess_45d_residuals_7d"):
            evaluated = (
                test_df
                .join(df_norm_train, on=["month", "day", "clim_day"], how="inner")
                .with_columns(
                    method=pl.lit(method),
                    test_year=pl.lit(test_year),
                )
                .with_columns(
                    z=(pl.col(param) - pl.col("mean_lowess_45d")) / pl.col(f"std_{method}"),
                )
                .with_columns(
                    gaussian_nll=pl.col(f"std_{method}").log() + 0.5 * pl.col("z").pow(2),
                )
                .sort("month", "day")
            )

            to_concat_evaluated.append(evaluated)

            scores_test_year = (
                evaluated
                .select(
                    pl.col("station_id"),
                    pl.col("station_name"),
                    pl.col("method"),
                    pl.col("test_year"),
                    n_targets=pl.col(param).count(),
                    n_scores=pl.col("z").is_finite().sum(),
                    mean_z=pl.col("z").mean(),
                    std_z=pl.col("z").std(),
                    abs_z_lt_1=(pl.col("z").abs() <= 1).sum() / (pl.col("z").abs() <= 1).count(),
                    abs_z_q68=pl.col("z").abs().quantile(0.68),
                    abs_z_lt_2=(pl.col("z").abs() <= 2).sum() / (pl.col("z").abs() <= 2).count(),
                    abs_z_q95=pl.col("z").abs().quantile(0.95),
                    abs_z_lt_3=(pl.col("z").abs() <= 3).sum() / (pl.col("z").abs() <= 3).count(),
                    abs_z_q99_7=pl.col("z").abs().quantile(0.997),
                    mean_nll=pl.col("gaussian_nll").mean(),
                    abs_z_max=pl.col("z").abs().max(),
                )
            )

            rows.append(scores_test_year.to_dicts()[0])


    cv_by_year = pl.DataFrame(rows)

    evaluated_concat: pl.DataFrame = pl.concat(to_concat_evaluated).sort("obs_date")

    # print(evaluated_concat)
    # print(
    #     evaluated_concat
    #     .group_by("station_id", "clim_day")
    #     .agg(
    #         pl.len(),
    #     )
    # )    

    cv_summary = (
        evaluated_concat
        # .group_by("method", "window")
        .group_by("station_id", "station_name", "method")
        .agg(
            n_targets=pl.col(param).count(),
            n_scores=pl.col("z").is_finite().sum(),
            mean_z=pl.col("z").mean(),
            std_z=pl.col("z").std(),
            abs_z_lt_1=(pl.col("z").abs() <= 1).sum() / (pl.col("z").abs() <= 1).count(),
            abs_z_q68=pl.col("z").abs().quantile(0.68),
            abs_z_lt_2=(pl.col("z").abs() <= 2).sum() / (pl.col("z").abs() <= 2).count(),
            abs_z_q95=pl.col("z").abs().quantile(0.95),
            abs_z_lt_3=(pl.col("z").abs() <= 3).sum() / (pl.col("z").abs() <= 3).count(),
            abs_z_q99_7=pl.col("z").abs().quantile(0.997),
            mean_nll=pl.col("gaussian_nll").mean(),
            abs_z_max=pl.col("z").abs().max(),
        )
        .sort("method")
    )

    cv_summaries.append(cv_summary)

    pl.concat(cv_summaries).write_csv(config.PROJECT_ROOT / "dump" / f"norm_method_test_{ts}.csv", separator=";")
    evaluated_concat.write_csv(config.PROJECT_ROOT / "dump" / f"norm_test_years_{ts}.csv", separator=";")

    with pl.Config(tbl_cols=35, tbl_rows=20, tbl_width_chars=400):
        print(cv_by_year)

    with pl.Config(tbl_cols=35, tbl_rows=50, tbl_width_chars=400):
        print(cv_summary)

shape: (60, 16)
┌────────────┬──────────────┬─────────────────────────┬───────────┬───────────┬──────────┬───────────┬──────────┬────────────┬───────────┬────────────┬───────────┬────────────┬─────────────┬──────────┬───────────┐
│ station_id ┆ station_name ┆ method                  ┆ test_year ┆ n_targets ┆ n_scores ┆ mean_z    ┆ std_z    ┆ abs_z_lt_1 ┆ abs_z_q68 ┆ abs_z_lt_2 ┆ abs_z_q95 ┆ abs_z_lt_3 ┆ abs_z_q99_7 ┆ mean_nll ┆ abs_z_max │
│ ---        ┆ ---          ┆ ---                     ┆ ---       ┆ ---       ┆ ---      ┆ ---       ┆ ---      ┆ ---        ┆ ---       ┆ ---        ┆ ---       ┆ ---        ┆ ---         ┆ ---      ┆ ---       │
│ str        ┆ str          ┆ str                     ┆ i64       ┆ i64       ┆ i64      ┆ f64       ┆ f64      ┆ f64        ┆ f64       ┆ f64        ┆ f64       ┆ f64        ┆ f64         ┆ f64      ┆ f64       │
╞════════════╪══════════════╪═════════════════════════╪═══════════╪═══════════╪══════════╪═══════════╪══════════╪════════════╪══